# DSO 576 Module 6 - Riley Lyu
Short-Term Rental Marketplace Health: Pricing Guidance Dashboard

Our project has three Los Angeles snapshots. For this cleaning assignment, I used the most recent snapshot, June 15, 2026, as a first example to inspect and clean the data. The same cleaning process can later be applied to the other snapshots.

Question: Which neighborhood + room type + bedroom segments warrant pricing review? Planned metric: segment median observed price with valid counts. This homework prepares data, not final pricing guidance.


## 1. Load Data
Inside Airbnb LA, June 15, 2026. Run from project/. Grain: one listing; one listing-date; one review; one neighborhood reference record. Source files are read only.


In [1]:
import pandas as pd

data_folder = "group data/2026-06-15/"
listings = pd.read_csv(data_folder + "listings.csv.gz", low_memory=False)
calendar = pd.read_csv(data_folder + "calendar.csv.gz", low_memory=False)
reviews = pd.read_csv(data_folder + "reviews.csv.gz", low_memory=False)
neighborhoods = pd.read_csv(data_folder + "neighbourhoods.csv", low_memory=False)


## 2. Inspect Data

In [2]:
names = ["Listings", "Calendar", "Reviews", "Neighborhoods"]
raw_tables = [listings, calendar, reviews, neighborhoods]
for name, table in [("Listings", listings), ("Calendar", calendar),
                    ("Reviews", reviews), ("Neighborhoods", neighborhoods)]:
    print(name, table.shape)
    print(table.columns)
    print(table.dtypes)
    print(table.isna().sum())
project_fields = ["id", "price", "neighbourhood_cleansed", "room_type", "bedrooms",
                  "property_type", "accommodates", "minimum_nights", "availability_90",
                  "number_of_reviews_ltm", "review_scores_value", "scrape_id", "last_scraped"]
for column in project_fields:
    assert column in listings.columns
print(listings[project_fields].head())


Listings (43751, 90)
Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_profile_id', 'host_profile_url', 'host_name',
       'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months',
       'hosts_time_as_host_years', 'hosts_time_as_host_months',
       'host_location', 'host_about', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood',
       'host_listings_count', 'host_total_listings_count',
       'host_verifications', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'price_quote_checkin_

### Cleaning decisions before applying them
| Field | Rule | Reason | Check |
|---|---|---|---|
| Price | Add numeric column; preserve original | Comparisons need numbers | Failures, missing counts, examples |
| Dates | Add parsed columns; preserve text | Support date filtering | Failures and date examples |
| Text | Check whitespace; keep clean labels | Preserve meaningful differences | Counts and unchanged examples |
| Duplicates | Keep unique records | No actual duplicates observed | Exact and key duplicates separately |
| Missing | Keep, do not fill | Unknown is not zero | Before/after counts |
| Suspicious | Keep, do not cap | Unusual does not prove wrong | Ranges and examples |


## 3. Clean Price and Dates

In [3]:
listings_clean = listings.copy()
calendar_clean = calendar.copy()
reviews_clean = reviews.copy()
neighborhoods_clean = neighborhoods.copy()
clean_tables = [listings_clean, calendar_clean, reviews_clean, neighborhoods_clean]

listings_clean["price_numeric"] = pd.to_numeric(
    listings_clean["price"].str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False), errors="coerce")
failures = (listings["price"].notna() & listings_clean["price_numeric"].isna()).sum()
print("Price failures:", failures)
print("Missing before/after:", listings["price"].isna().sum(),
      listings_clean["price_numeric"].isna().sum())
print(listings_clean[["id", "price", "price_numeric"]].head())
assert failures == 0
assert (listings["price"].isna() != listings_clean["price_numeric"].isna()).sum() == 0


Price failures: 0
Missing before/after: 5565 5565
     id    price  price_numeric
0  2708   $67.22          67.22
1  2732  $213.56         213.56
2  6033   $99.63          99.63
3  6931   $95.21          95.21
4  7874  $113.00         113.00


In [4]:
for name, table, original, parsed in [
    ("Listings", listings_clean, "last_scraped", "last_scraped_dt"),
    ("Calendar", calendar_clean, "date", "date_dt"),
    ("Reviews", reviews_clean, "date", "date_dt")
]:
    table[parsed] = pd.to_datetime(table[original], format="%Y-%m-%d", errors="coerce")
    failures = (table[original].notna() & table[parsed].isna()).sum()
    print(name, "date failures:", failures)
    print("Range:", table[parsed].min(), table[parsed].max())
    assert failures == 0
print(listings_clean[["last_scraped", "last_scraped_dt"]].head())


Listings date failures: 0
Range: 2026-06-15 00:00:00 2026-06-23 00:00:00
Calendar date failures: 0
Range: 2026-06-15 00:00:00 2027-06-22 00:00:00
Reviews date failures: 0
Range: 2009-05-26 00:00:00 2026-06-22 00:00:00
  last_scraped last_scraped_dt
0   2026-06-15      2026-06-15
1   2026-06-16      2026-06-16
2   2026-06-22      2026-06-22
3   2026-06-15      2026-06-15
4   2026-06-16      2026-06-16


## 4. Check Text

In [5]:
for column in ["neighbourhood_cleansed", "room_type", "property_type"]:
    print(column, "whitespace:", (listings[column] != listings[column].str.strip()).sum())
    print(pd.DataFrame({"before": listings[column].head(),
                        "after": listings_clean[column].head()}))
print("Reference whitespace:", (neighborhoods["neighbourhood"] != neighborhoods["neighbourhood"].str.strip()).sum())


neighbourhood_cleansed whitespace: 0
           before           after
0       Hollywood       Hollywood
1    Santa Monica    Santa Monica
2  Woodland Hills  Woodland Hills
3       Hollywood       Hollywood
4      Bellflower      Bellflower
room_type whitespace: 0
            before            after
0     Private room     Private room
1     Private room     Private room
2  Entire home/apt  Entire home/apt
3     Private room     Private room
4     Private room     Private room
property_type whitespace: 0
                        before                        after
0  Private room in rental unit  Private room in rental unit
1  Private room in rental unit  Private room in rental unit
2              Entire bungalow              Entire bungalow
3  Private room in rental unit  Private room in rental unit
4         Private room in home         Private room in home
Reference whitespace: 0


## 5. Check Duplicates
Calendar's key is listing_id + date, not listing_id alone. No rows removed.


In [6]:
keys = [["id"], ["listing_id", "date"], ["id"], ["neighbourhood"]]
for name, table, key in [
    ("Listings", listings_clean, ["id"]),
    ("Calendar", calendar_clean, ["listing_id", "date"]),
    ("Reviews", reviews_clean, ["id"]),
    ("Neighborhoods", neighborhoods_clean, ["neighbourhood"])
]:
    print(name, "exact duplicates:", table.duplicated().sum())
    print("Missing keys:", len(table.loc[table[key[0]].isna()]))
    print("Duplicate keys:", table.duplicated(key).sum())
    assert len(table.loc[table[key[0]].isna()]) == 0
    assert table.duplicated(key).sum() == 0

assert calendar_clean["date"].isna().sum() == 0


Listings exact duplicates: 0
Missing keys: 0
Duplicate keys: 0
Calendar exact duplicates: 0
Missing keys: 0
Duplicate keys: 0
Reviews exact duplicates: 0
Missing keys: 0
Duplicate keys: 0
Neighborhoods exact duplicates: 0
Missing keys: 0
Duplicate keys: 0


## 6. Check Missing Values
Retain missing values. Missing bedroom is not zero; missing rating is not a low score. Reviews with missing text are retained.


In [7]:
for name, raw, clean in [
    ("Listings", listings, listings_clean), ("Calendar", calendar, calendar_clean),
    ("Reviews", reviews, reviews_clean), ("Neighborhoods", neighborhoods, neighborhoods_clean)
]:
    print(name)
    print(pd.DataFrame({"before": raw.isna().sum(),
                        "after": clean[raw.columns].isna().sum()}))


Listings
                                              before  after
id                                                 0      0
listing_url                                        0      0
scrape_id                                          0      0
last_scraped                                       0      0
source                                             0      0
...                                              ...    ...
calculated_host_listings_count                     0      0
calculated_host_listings_count_entire_homes        0      0
calculated_host_listings_count_private_rooms       0      0
calculated_host_listings_count_shared_rooms        0      0
reviews_per_month                              11257  11257

[90 rows x 2 columns]
Calendar
                before  after
listing_id           0      0
date                 0      0
available            0      0
minimum_nights       0      0
maximum_nights       0      0
Reviews
               before  after
listing_id          0 

## 7. Check Suspicious Values
Retain extremes and zero scores. Prior investigation found monthly-quote ambiguity, 181 Calendar-only IDs (66,065 rows) and 14 extended windows. No merge, price correction or date trimming is justified yet.


In [8]:
for column in ["price_numeric", "bedrooms", "minimum_nights",
               "review_scores_value", "availability_90", "accommodates"]:
    print(column, "min:", listings_clean[column].min(), "max:", listings_clean[column].max())
print("Nonpositive price:", (listings_clean["price_numeric"] <= 0).sum())
print("Negative bedrooms:", (listings_clean["bedrooms"] < 0).sum())
print("Nonpositive minimum stay:", (listings_clean["minimum_nights"] <= 0).sum())
print("Scores outside 0-5:", ((listings_clean["review_scores_value"] < 0)
                             | (listings_clean["review_scores_value"] > 5)).sum())
print("Availability outside 0-90:", ((listings_clean["availability_90"] < 0)
                                   | (listings_clean["availability_90"] > 90)).sum())
print("Nonpositive capacity:", (listings_clean["accommodates"] <= 0).sum())
print(listings_clean.loc[(listings_clean["review_scores_value"] == 0),
                        ["id", "review_scores_value", "number_of_reviews"]])


price_numeric min: 3.77 max: 20583.33
bedrooms min: 0.0 max: 24.0
minimum_nights min: 1.0 max: 1124.0
review_scores_value min: 0.0 max: 5.0
availability_90 min: 0 max: 90
accommodates min: 1 max: 16
Nonpositive price: 0
Negative bedrooms: 0
Nonpositive minimum stay: 0
Scores outside 0-5: 0
Availability outside 0-90: 0
Nonpositive capacity: 0
           id  review_scores_value  number_of_reviews
1927  6764023                  0.0                  1


### Calculation counts
Means and medians skip missing values; missing values are not zero. No final segment benchmark is produced until quote comparability is resolved.


In [9]:
print("Valid prices:", listings_clean["price_numeric"].notna().sum())
print("Price median:", listings_clean["price_numeric"].median())
print("Valid scores:", listings_clean["review_scores_value"].notna().sum())
print("Score mean, retaining zero:", listings_clean["review_scores_value"].mean())


Valid prices: 38186
Price median: 223.67
Valid scores: 32485
Score mean, retaining zero: 4.71149484377405


## 8. Check Row Counts

In [10]:
row_counts = pd.DataFrame({
    "table": names,
    "starting_rows": [len(listings), len(calendar), len(reviews), len(neighborhoods)],
    "final_rows": [len(listings_clean), len(calendar_clean), len(reviews_clean), len(neighborhoods_clean)]
})
row_counts["change"] = row_counts["final_rows"] - row_counts["starting_rows"]
print(row_counts)
assert (row_counts["change"] != 0).sum() == 0
print("Cleaning uses copies and only adds new columns; no source columns are overwritten.")


           table  starting_rows  final_rows  change
0       Listings          43751       43751       0
1       Calendar       16035198    16035198       0
2        Reviews        1856294     1856294       0
3  Neighborhoods            270         270       0
Cleaning uses copies and only adds new columns; no source columns are overwritten.


## 9. Five Record Checks
Expected results and reasons, stated first: 2708 normal price -> 67.22; 2436587 unknown minimum_nights stays missing; 6764023 zero score stays zero; 25870842 maximum price stays 20583.33; 607365253103904668 minimum price stays 3.77. No capping or imputation.


In [11]:
# Expectations were stated above before checking these five records.
original1 = listings.loc[listings["id"] == 2708, "price"].iloc[0]
actual1 = listings_clean.loc[listings_clean["id"] == 2708, "price_numeric"].iloc[0]

original2 = listings.loc[listings["id"] == 2436587, "minimum_nights"]
actual2 = listings_clean.loc[listings_clean["id"] == 2436587, "minimum_nights"]
missing_match = (original2.isna().sum() == 1) and (actual2.isna().sum() == 1)
print("Missing minimum_nights before and after:")
print(original2)
print(actual2)

original3 = listings.loc[listings["id"] == 6764023, "review_scores_value"].iloc[0]
actual3 = listings_clean.loc[listings_clean["id"] == 6764023, "review_scores_value"].iloc[0]

original4 = listings.loc[listings["id"] == 25870842, "price"].iloc[0]
actual4 = listings_clean.loc[listings_clean["id"] == 25870842, "price_numeric"].iloc[0]

original5 = listings.loc[listings["id"] == 607365253103904668, "price"].iloc[0]
actual5 = listings_clean.loc[listings_clean["id"] == 607365253103904668, "price_numeric"].iloc[0]

record_checks = pd.DataFrame({
    "ID": [2708, 2436587, 6764023, 25870842, 607365253103904668],
    "Field": ["price_numeric", "minimum_nights", "review_scores_value", "price_numeric", "price_numeric"],
    "Original": [original1, original2.iloc[0], original3, original4, original5],
    "Expected": [67.22, "Missing", 0.0, 20583.33, 3.77],
    "Actual": [actual1, actual2.iloc[0], actual3, actual4, actual5],
    "Match": [actual1 == 67.22, missing_match, actual3 == 0.0,
              actual4 == 20583.33, actual5 == 3.77]
})
print(record_checks)
assert record_checks["Match"].sum() == 5
print("NaN in the table means Missing.")
print("Codex price suggestion checked: examples matched; full-column failures 0; missing count unchanged.")


Missing minimum_nights before and after:
919   NaN
Name: minimum_nights, dtype: float64
919   NaN
Name: minimum_nights, dtype: float64
                   ID                Field    Original  Expected    Actual  \
0                2708        price_numeric      $67.22     67.22     67.22   
1             2436587       minimum_nights         NaN   Missing       NaN   
2             6764023  review_scores_value         0.0       0.0      0.00   
3            25870842        price_numeric  $20,583.33  20583.33  20583.33   
4  607365253103904668        price_numeric       $3.77      3.77      3.77   

   Match  
0   True  
1   True  
2   True  
3   True  
4   True  
NaN in the table means Missing.
Codex price suggestion checked: examples matched; full-column failures 0; missing count unchanged.


## 10. Create 50-Row Sample
Select the five checked records, six missing-value examples and six unusual-value examples, then ordinary records to reach 50 unique rows. Some selections overlap. Deduplicate only the sample, never the source data.


In [12]:
check_ids = [2708, 2436587, 6764023, 25870842, 607365253103904668]
missing_cases = listings_clean.loc[
    listings_clean["price_numeric"].isna()
    | listings_clean["bedrooms"].isna()
    | listings_clean["minimum_nights"].isna()
    | listings_clean["review_scores_value"].isna()
].head(6)
unusual_cases = listings_clean.loc[
    (listings_clean["bedrooms"] == 0)
    | (listings_clean["bedrooms"] >= 10)
    | (listings_clean["minimum_nights"] >= 365)
].head(6)
sample = pd.concat([
    listings_clean.loc[listings_clean["id"].isin(check_ids)],
    missing_cases,
    unusual_cases,
    listings_clean.head(50)
])
# Remove repeated selections from this sample only.
sample = sample.loc[~sample["id"].duplicated()].head(50)
sample = sample[project_fields + ["price_numeric", "last_scraped_dt"]]
assert len(sample) == 50
assert sample["id"].duplicated().sum() == 0
assert sample["id"].isin(check_ids).sum() == 5
sample.to_csv("cleaning_outputs/cleaned_listings_sample_50.csv", index=False)
print("Saved 50 sample rows; no full dataset exported.")


Saved 50 sample rows; no full dataset exported.


## 11. Reproducibility Check
From project/, choose the .venv kernel, Restart Kernel and Run All. The script repeats the same code and exports only the sample. No original source file is written.

Checklist: 1 vision in introduction; 2 grains/keys in 1/5; 3 preserve sources in 1/8; 4 types in 2/3; 5 text in 4; 6 duplicates in 5; 7 missing in 6; 8 suspicious in 7; 9 merges Not applicable; 10 row reconciliation in 8; 11 calculations in 7; 12 five records in 9; 13 restart/run in 11; 14 limitations in 7 and report.


In [13]:
assert (row_counts["change"] != 0).sum() == 0
assert record_checks["Match"].sum() == 5
print("All checks passed. Only the sample CSV is exported.")


All checks passed. Only the sample CSV is exported.
